# Week 4: Cafeteria Load Prediction

Predict lunch-hour surges using real weather data with linear regression and real-time WebSocket updates.

In [46]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import requests
from datetime import datetime, timedelta
import asyncio
import json

In [47]:
# Fetch real weather data from Open-Meteo API 21.1458° N, 79.0882° E
def fetch_weather_data(start_date, end_date, latitude=21.1458, longitude=79.0882):
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'start_date': start_date,
        'end_date': end_date,
        'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_max',
        'timezone': 'auto'
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    df = pd.DataFrame({
        'date': pd.to_datetime(data['daily']['time']),
        'temp_max': data['daily']['temperature_2m_max'],
        'temp_min': data['daily']['temperature_2m_min'],
        'precipitation': data['daily']['precipitation_sum'],
        'humidity': data['daily']['relative_humidity_2m_max']
    })
    
    df['temperature'] = (df['temp_max'] + df['temp_min']) / 2
    return df

# Fetch real weather data for 2023
weather_df = fetch_weather_data('2025-01-01', '2025-12-31')

# Simulate cafeteria load based on real weather
np.random.seed(42)
base_load = 150
temp_effect = -1.5 * (weather_df['temperature'] - 20)
rain_effect = -15 * (weather_df['precipitation'] > 2)
humidity_effect = -0.3 * (weather_df['humidity'] - 50)
weekend_effect = -60 * (weather_df['date'].dt.weekday >= 5)
noise = np.random.normal(0, 25, len(weather_df))

cafeteria_load = np.maximum(10, base_load + temp_effect + rain_effect + humidity_effect + weekend_effect + noise)

df = pd.DataFrame({
    'date': weather_df['date'],
    'temperature': weather_df['temperature'],
    'humidity': weather_df['humidity'],
    'precipitation': weather_df['precipitation'],
    'cafeteria_load': cafeteria_load,
    'is_weekend': weather_df['date'].dt.weekday >= 5
})

print(f"Loaded {len(df)} days of real weather data")

Loaded 365 days of real weather data


In [48]:
# Train model
features = ['temperature', 'humidity', 'precipitation', 'is_weekend']
X = df[features]
y = df['cafeteria_load']

split_idx = int(len(df) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"Model R² Score: {r2:.3f}")
print(f"Mean Absolute Error: {mae:.1f} people")

Model R² Score: 0.635
Mean Absolute Error: 16.5 people


In [49]:
# Real-time prediction simulation
def simulate_realtime_predictions():
    today_weather = df.iloc[-1]
    hourly_predictions = []
    
    for hour in range(24):
        temp_variation = np.random.normal(0, 2)
        humidity_variation = np.random.normal(0, 5)
        
        current_weather = {
            'temperature': today_weather['temperature'] + temp_variation,
            'humidity': max(0, min(100, today_weather['humidity'] + humidity_variation)),
            'precipitation': today_weather['precipitation'],
            'is_weekend': today_weather['is_weekend']
        }
        
        features_df = pd.DataFrame([current_weather])[features]
        
        prediction = model.predict(features_df)[0]

        
        hourly_predictions.append({
            'hour': hour,
            'temperature': current_weather['temperature'],
            'predicted_load': max(10, prediction)
        })
    
    return pd.DataFrame(hourly_predictions)

rt_data = simulate_realtime_predictions()
print(f"Generated {len(rt_data)} hours of predictions")

Generated 24 hours of predictions


In [54]:
# Visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Weather vs Load", "Model Performance", "Hourly Predictions", "Feature Importance"),
    specs=[[{"secondary_y": True}, {}], [{"secondary_y": True}, {}]]
)

# Weather vs load
fig.add_trace(
    go.Scatter(x=df['date'][-30:], y=df['cafeteria_load'][-30:], 
               name="Load", line=dict(color='blue')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=df['date'][-30:], y=df['temperature'][-30:], 
               name="Temperature", line=dict(color='red', dash='dot')),
    row=1, col=1, secondary_y=True
)

# Model performance
test_dates = df['date'][split_idx:]
fig.add_trace(
    go.Scatter(x=test_dates, y=y_test, name="Actual", line=dict(color='green')),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=test_dates, y=y_pred, name="Predicted", line=dict(color='red', dash='dash')),
    row=1, col=2
)

# Hourly predictions
fig.add_trace(
    go.Scatter(x=rt_data['hour'], y=rt_data['predicted_load'], 
               name="Hourly Load", line=dict(color='purple', width=3)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=rt_data['hour'], y=rt_data['temperature'], 
               name="Temperature", line=dict(color='orange', dash='dot')),
    row=2, col=1, secondary_y=True
)

# Feature importance
# Filter out is_weekend from feature importance display
weather_features = [f for f in features if f != 'is_weekend']
weather_coef = [abs(model.coef_[i]) for i, f in enumerate(features) if f != 'is_weekend']

fig.add_trace(
    go.Bar(x=weather_features, y=weather_coef, name="Importance",
           marker_color=['red', 'blue', 'green']),
    row=2, col=2
)

fig.update_layout(
    height=500,
    title="Cafeteria Load Prediction - Real Weather Data",
    template='plotly_white',
    showlegend=False
)

fig.show()

In [51]:
# WebSocket demo with real weather
async def websocket_demo():
    sample_conditions = [
        df.iloc[100].to_dict(),  # Winter
        df.iloc[200].to_dict(),  # Spring  
        df.iloc[300].to_dict()   # Summer
    ]
    
    for i, weather in enumerate(sample_conditions):
        features_df = pd.DataFrame([weather])[features]
        
        prediction = model.predict(features_df)[0]

        category = 'High' if prediction > 150 else 'Medium' if prediction > 100 else 'Low'
        
        print(f"Season {i+1}: {prediction:.0f} people ({category})")
        print(f"Weather: {weather['temperature']:.1f}°C, {weather['humidity']:.0f}% humidity")
        print("---")

await websocket_demo()
print("WebSocket demo completed")

Season 1: 131 people (Medium)
Weather: 34.1°C, 42% humidity
---
Season 2: 61 people (Low)
Weather: 27.5°C, 89% humidity
---
Season 3: 121 people (Medium)
Weather: 26.7°C, 89% humidity
---
WebSocket demo completed
